# Step 07A — Clinical error figures from FROZEN validator

**Purpose:** Create lesion-level TP, FP, and FN image overlays that match the verified Step 01B fixed-confidence validator, not a separate `predict()` pipeline. No retraining and no optimization.

**Frozen inputs:** YOLO11s model SHA-256 `19ce0e2ae9731a9acb5baeaa4cb514782aa182e7d916d8a101298dd59cdb077e`; internal 30 radiographs / 108 source-defined boxes; external 66 / 55. Confidence = **0.25**, NMS IoU = **0.70**, lesion matching IoU = **0.50**, original scale.

**Ethical interpretation:** TP/FP/FN are defined **only against the original source labels**. Two experts flagged 11 source-label-negative radiographs as containing possible caries. An unmatched prediction may therefore *not* be a clinically false detection. Do not describe these examples as independently adjudicated clinical truth.

The notebook asserts the official per-image TP/FP/FN counts before producing the publication panels.


In [ ]:
%pip -q install "ultralytics==8.4.139" pillow pyyaml pandas numpy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import torch, ultralytics, numpy as np, pandas as pd, yaml, hashlib, json, zipfile, math, shutil
from datetime import datetime,timezone
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
WEIGHTS=ROOT/'final_model/YOLO11s_Final/final_60epochs.pt'
DATA={'internal':ROOT/'processed_data/yolo_caries_baseline',
      'external':ROOT/'external_validation/MOPG7_66'}
EXPECTED_HASH='19ce0e2ae9731a9acb5baeaa4cb514782aa182e7d916d8a101298dd59cdb077e'
EXPECTED_COUNTS={'internal':(30,108),'external':(66,55)}
DEVICE=0 if torch.cuda.is_available() else 'cpu'
def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
assert WEIGHTS.exists() and sha256(WEIGHTS)==EXPECTED_HASH,'STOP: Checkpoint mismatch.'
assert ultralytics.__version__=='8.4.139',ultralytics.__version__
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'Step_07A_Validator_Figures_{STAMP}'
OUT.mkdir(parents=True, exist_ok=False)
(OUT/'panels').mkdir()
manifest=[]
for cohort,folder in DATA.items():
 ips=sorted([p for p in (folder/'images/test').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')])
 n_gt=0
 for ip in ips:
  lp=folder/'labels/test'/f'{ip.stem}.txt'
  assert lp.exists(),lp
  lines=[q.strip() for q in lp.read_text().splitlines() if q.strip()]
  n_gt+=len(lines)
  manifest.append({'cohort':cohort,'image':ip.name,'image_sha256':sha256(ip),'label_sha256':sha256(lp),'n_gt':len(lines)})
 assert (len(ips),n_gt)==EXPECTED_COUNTS[cohort],(cohort,len(ips),n_gt)
 spec={'path':str(folder),'train':'images/test','val':'images/test','test':'images/test','names':{0:'caries'},'nc':1}
 (OUT/f'{cohort}.yaml').write_text(yaml.safe_dump(spec,sort_keys=False))
pd.DataFrame(manifest).to_csv(OUT/'input_manifest_sha256.csv',index=False)
print('PASS: checkpoint + cohort image/GT counts. Device:', DEVICE,'Output:',OUT)

In [ ]:
EXPECTED_FIXED = {'external': {'nir105.jpg': [0, 0, 1, 0], 'nir112.jpg': [0, 0, 2, 0], 'nir20.jpg': [0, 0, 2, 0], '0000754.jpg': [0, 0, 2, 0], '0000665.jpg': [0, 0, 0, 0], '0000538.jpg': [0, 0, 0, 0], '00001480.jpg': [0, 0, 0, 0], '00001099.jpg': [0, 0, 1, 0], '0000544.jpg': [0, 0, 1, 0], '0000459.jpg': [0, 0, 1, 0], '0000460.jpg': [0, 0, 1, 0], '00001457.jpg': [0, 0, 1, 0], '0000121.jpg': [0, 0, 1, 0], '00001378.jpg': [0, 0, 1, 0], '00001381.jpg': [0, 1, 0, 1], '00001205.jpg': [0, 0, 0, 0], '00001373.jpg': [0, 0, 0, 0], '00001306.jpg': [0, 0, 1, 0], '00001467.jpg': [0, 2, 0, 2], '00001396.jpg': [0, 0, 1, 0], '00001421.jpg': [0, 0, 0, 0], '00001216.jpg': [0, 0, 2, 0], '00001268.jpg': [0, 1, 0, 1], '00001247.jpg': [0, 0, 0, 0], '000077.jpg': [0, 0, 0, 0], '00001202.jpg': [0, 0, 2, 0], '00001209.jpg': [0, 1, 0, 1], '0000689.jpg': [0, 0, 0, 0], '0000727.jpg': [0, 0, 1, 0], '00001529.jpg': [0, 0, 2, 0], '00001020.jpg': [0, 0, 0, 0], '0000287.jpg': [0, 0, 4, 0], '0000697.jpg': [0, 0, 1, 0], '00001132.jpg': [0, 0, 2, 0], '00001535.jpg': [0, 0, 0, 0], '0000862.jpg': [0, 0, 0, 0], '0000588.jpg': [0, 0, 1, 0], '0000847.jpg': [0, 0, 1, 0], '0000614.jpg': [0, 0, 0, 0], '0000555.jpg': [0, 0, 1, 0], '00001477.jpeg': [0, 0, 1, 0], '0000541.jpg': [0, 0, 1, 0], '00001266.jpg': [0, 0, 1, 0], '0000837.jpg': [0, 0, 1, 0], '0000717.jpg': [0, 0, 0, 0], '0000701.jpg': [0, 0, 2, 0], '0000623.jpg': [0, 0, 0, 0], '00001492.jpg': [0, 0, 3, 0], 'ibn59.jpg': [0, 0, 0, 0], 'ibn51.jpg': [0, 0, 1, 0], 'ibn520.jpg': [0, 1, 0, 1], 'ibn211.jpg': [0, 0, 1, 0], 'ibn451.jpg': [0, 0, 2, 0], 'ibn404.jpg': [0, 0, 1, 0], 'ibn342.jpg': [0, 0, 2, 0], 'ibn285.jpg': [0, 0, 0, 0], 'ibn206.jpg': [0, 0, 2, 0], 'ibn159.jpg': [0, 0, 0, 0], 'ibn179.jpg': [0, 0, 0, 0], 'ibn324.jpg': [0, 0, 0, 0], 'ibn480.jpg': [0, 0, 1, 0], 'ibn504.jpg': [0, 0, 0, 0], 'ibn326.jpg': [0, 0, 1, 0], 'ibn115.jpg': [0, 0, 1, 0], 'ibn99.jpg': [0, 0, 1, 0], 'ibn68.jpg': [0, 0, 0, 0]}, 'internal': {'cate1_000.png': [0, 0, 0, 0], 'cate1_001.png': [0, 0, 3, 0], 'cate1_002.png': [1, 0, 6, 1], 'cate1_003.png': [0, 0, 0, 0], 'cate1_004.png': [1, 0, 7, 1], 'cate1_005.png': [0, 0, 2, 0], 'cate1_006.png': [2, 2, 2, 4], 'cate1_007.png': [1, 0, 8, 1], 'cate1_008.png': [1, 0, 3, 1], 'cate1_009.png': [0, 0, 0, 0], 'cate1_010.png': [0, 0, 0, 0], 'cate1_011.png': [0, 0, 8, 0], 'cate1_012.png': [0, 0, 1, 0], 'cate1_013.png': [0, 0, 7, 0], 'cate1_014.png': [0, 0, 2, 0], 'cate2_000.png': [0, 0, 2, 0], 'cate2_001.png': [0, 0, 2, 0], 'cate2_002.png': [0, 0, 5, 0], 'cate2_003.png': [0, 0, 5, 0], 'cate2_004.png': [0, 0, 5, 0], 'cate2_005.png': [0, 1, 2, 1], 'cate2_006.png': [0, 0, 9, 0], 'cate2_007.png': [0, 0, 1, 0], 'cate2_008.png': [0, 0, 5, 0], 'cate2_009.png': [0, 0, 5, 0], 'cate2_010.png': [0, 0, 0, 0], 'cate2_011.png': [3, 1, 7, 4], 'cate2_012.png': [0, 2, 1, 2], 'cate2_013.png': [0, 0, 0, 0], 'cate2_014.png': [0, 0, 1, 0]}}
print("Loaded 96 previously validated image-level TP/FP/FN records.")

In [ ]:
# Re-run ONLY the official model.val() fixed-threshold pathway, requesting its own exported prediction boxes.
# IMPORTANT: keep the settings aligned with Step 01B; do not call model.predict().
OFFICIAL={}
for cohort in DATA:
 model=YOLO(str(WEIGHTS))
 metrics=model.val(data=str(OUT/f'{cohort}.yaml'),split='test',imgsz=1024,conf=.25,
                   iou=.70,max_det=300,device=DEVICE,half=False,workers=0,batch=4,
                   save=False,save_txt=True,save_conf=True,save_json=False,plots=False,
                   project=str(OUT/'validation_runs'),name=f'{cohort}_fixed',exist_ok=False,verbose=False)
 OFFICIAL[cohort]={'mAP50':float(metrics.box.map50),'mAP50_95':float(metrics.box.map),
                   'save_dir':str(metrics.save_dir)}
 print(cohort,OFFICIAL[cohort])
assert all(Path(v['save_dir']).exists() for v in OFFICIAL.values())
(OUT/'official_validation.json').write_text(json.dumps(OFFICIAL,indent=2))

In [ ]:
# Read source GT and validator-exported predictions in normalized YOLO format.
# Validation writes predictions to its own labels subfolder; absent file means zero predictions.
def read_yolo(p,W,H,is_pred=False):
 arr=[]
 if not p.exists():return arr
 for line in p.read_text().splitlines():
  if not line.strip():continue
  vals=[float(q) for q in line.split()]
  assert len(vals)==(6 if is_pred else 5),(p,line)
  cls,xc,yc,bw,bh=vals[:5]
  assert int(cls)==0,(p,line)
  x1=(xc-bw/2)*W;y1=(yc-bh/2)*H;x2=(xc+bw/2)*W;y2=(yc+bh/2)*H
  arr.append({'box':np.asarray([x1,y1,x2,y2],dtype=float),
              'score':float(vals[5]) if is_pred else None})
 return arr

def iou(boxa,boxb):
 x1=max(boxa[0],boxb[0]);y1=max(boxa[1],boxb[1]);x2=min(boxa[2],boxb[2]);y2=min(boxa[3],boxb[3])
 inter=max(0,x2-x1)*max(0,y2-y1)
 ar1=max(0,boxa[2]-boxa[0])*max(0,boxa[3]-boxa[1])
 ar2=max(0,boxb[2]-boxb[0])*max(0,boxb[3]-boxb[1])
 den=ar1+ar2-inter
 return inter/den if den>0 else 0.0

def match_one_to_one(pred,gt,threshold=.5):
 # One-class, IoU-priority matching. Independent of test-label modifications.
 candidates=sorted([(iou(p['box'],g['box']),pi,gi)
   for pi,p in enumerate(pred) for gi,g in enumerate(gt)],reverse=True)
 matched_pred={};used_gt=set()
 for ov,pi,gi in candidates:
  if ov<threshold:break
  if pi not in matched_pred and gi not in used_gt:
   matched_pred[pi]=(gi,ov);used_gt.add(gi)
 return matched_pred,used_gt

rows=[];errors=[];images={}
for cohort,folder in DATA.items():
 for filename,target in EXPECTED_FIXED[cohort].items():
  ip=folder/'images/test'/filename
  with Image.open(ip) as im: W,H=im.size
  gt=read_yolo(folder/'labels/test'/f'{ip.stem}.txt',W,H)
  pp=Path(OFFICIAL[cohort]['save_dir'])/'labels'/f'{ip.stem}.txt'
  pred=read_yolo(pp,W,H,is_pred=True)
  matched,used_gt=match_one_to_one(pred,gt)
  counts=[len(matched),len(pred)-len(matched),len(gt)-len(used_gt),len(pred)]
  if counts!=target: errors.append({'cohort':cohort,'image':filename,'expected':target,'got':counts,'pred_path':str(pp)})
  rows.append({'cohort':cohort,'image':filename,'TP':counts[0],'FP':counts[1],'FN':counts[2],
               'n_predictions':counts[3],'n_gt':len(gt),'expected_TP':target[0],
               'expected_FP':target[1],'expected_FN':target[2],'match_exact':counts==target})
  images[(cohort,filename)]=(ip,gt,pred,matched,used_gt)
df=pd.DataFrame(rows)
df.to_csv(OUT/'all_96_images_matching_audit.csv',index=False)
(OUT/'matching_discrepancies.json').write_text(json.dumps(errors,indent=2))
print('Image-level exact matches:',int(df.match_exact.sum()),'of',len(df))
print(df.groupby('cohort')[['TP','FP','FN','n_predictions','n_gt']].sum())
# Do not publish figures unless every per-image count reproduces the Step 01B validator.
assert not errors, 'STOP: Exported validator boxes and frozen Step 01B per-image counts disagree. Upload the audit ZIP and do not use generated panels.'
assert df.query("cohort=='internal'")[['TP','FP','FN']].sum().tolist()==[9,6,99]
assert df.query("cohort=='external'")[['TP','FP','FN']].sum().tolist()==[0,6,55]
print('PASS: 96/96 per-image lesion matching counts align with Step 01B.')

In [ ]:
# Create publication candidate panels. ALL classifications refer to source-defined GT, not independently corrected clinical truth.
CANDIDATES={
 'internal':['cate1_006.png','cate2_011.png','cate2_005.png','cate1_011.png'],
 'external':['0000287.jpg','00001467.jpg','ibn520.jpg']}
CLINICIAN_FLAGGED_SOURCE_NEGATIVES={
 'cate1_009.png','cate1_010.png','cate2_010.png',
 '00001247.jpg','00001373.jpg','00001381.jpg','0000538.jpg','0000623.jpg',
 'ibn159.jpg','ibn285.jpg','ibn59.jpg'}
# Source annotation overlay legend: green=matched prediction; orange=unmatched prediction; red=matched reference;
# cyan dashed=unmatched source reference lesion. Text on predictions includes confidence and matched IoU.
try:
 FONT=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',22)
 SMALL=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',17)
except OSError:
 FONT=ImageFont.load_default();SMALL=FONT

def dashed_rectangle(draw,xy,color,width=4,dash=15,gap=9):
 x1,y1,x2,y2=map(int,xy)
 def seg(xa,ya,xb,yb):
  dx=xb-xa;dy=yb-ya; length=max(1,math.hypot(dx,dy));dist=0
  while dist<length:
   e=min(length,dist+dash)
   draw.line([(xa+dx*dist/length,ya+dy*dist/length),(xa+dx*e/length,ya+dy*e/length)],fill=color,width=width)
   dist+=dash+gap
 for e in [(x1,y1,x2,y1),(x2,y1,x2,y2),(x2,y2,x1,y2),(x1,y2,x1,y1)]:seg(*e)

def fig_one(cohort,name):
 ip,gt,pred,matched,used_gt=images[(cohort,name)]
 im=Image.open(ip).convert('RGB')
 W,H=im.size; pad_top=100; pad_bottom=60
 canvas=Image.new('RGB',(W,H+pad_top+pad_bottom),'#f6f7f8')
 canvas.paste(im,(0,pad_top));d=ImageDraw.Draw(canvas)
 n_tp=len(matched);n_fp=len(pred)-n_tp;n_fn=len(gt)-len(used_gt)
 flag=' | EXPERT REVIEW DISAGREES WITH SOURCE-NEGATIVE LABEL' if name in CLINICIAN_FLAGGED_SOURCE_NEGATIVES else ''
 d.text((18,12),f'{cohort.upper()} — {name} — Source-GT TP={n_tp}, FP={n_fp}, FN={n_fn}',font=FONT,fill='#131313')
 d.text((18,46),f'Confidence ≥0.25 | IoU ≥0.50 | frozen checkpoint{flag}',font=SMALL,fill='#222222')
 for gi,g in enumerate(gt):
  box=g['box'].copy();box[[1,3]]+=pad_top
  if gi in used_gt:d.rectangle(tuple(box),outline='#df2848',width=4)
  else:dashed_rectangle(d,box,'#00b5cc',width=4)
 for pi,p in enumerate(pred):
  b=p['box'].copy();b[[1,3]]+=pad_top
  if pi in matched:
   label=f'P{pi+1} TP conf={p["score"]:.3f} IoU={matched[pi][1]:.3f}';col='#15a661'
  else:
   label=f'P{pi+1} unmatched conf={p["score"]:.3f}';col='#f4a340'
  d.rectangle(tuple(b),outline=col,width=5)
  tx=max(8,min(int(b[0]),W-570));ty=max(pad_top,min(int(b[1])-24,H+pad_top-25))
  d.rectangle((tx,ty, min(W-8,tx+min(570,len(label)*11)),ty+24),fill='#141414')
  d.text((tx+4,ty+1),label,font=SMALL,fill=col)
 d.text((18,H+pad_top+16),'Reference matched: RED | Reference missed: CYAN DASHED | Prediction matched: GREEN | Prediction unmatched: ORANGE',font=SMALL,fill='#222222')
 output=OUT/'panels'/f'{cohort}_{ip.stem}_source_GT.png'
 canvas.save(output)
 return output
panels=[]
for cohort,names in CANDIDATES.items():
 for n in names:
  p=fig_one(cohort,n);panels.append(p);print('Saved',p.name)
# Make overview contact sheet for choosing examples. Do not use tiny overview as a publication resolution figure.
thumbs=[]
for p in panels:
 with Image.open(p) as img:
  copied=img.copy();copied.thumbnail((1250,650));thumbs.append(copied)
cols=2;rows_count=(len(thumbs)+cols-1)//cols;cw=1270;ch=670
montage=Image.new('RGB',(cols*cw,rows_count*ch),'white')
for i,img in enumerate(thumbs):montage.paste(img,((i%cols)*cw,(i//cols)*ch))
montage.save(OUT/'Figure_Clinical_Cases_Contact_Sheet.png')
print('Generated',len(panels),'audited publication-size panels.')

In [ ]:
# Export evidence, plus a file explaining the annotation caveat and licensing requirement.
readme = """STEP 07A — FIGURE GENERATION FROM FROZEN VALIDATOR

Checkpoint SHA256: 19ce0e2ae9731a9acb5baeaa4cb514782aa182e7d916d8a101298dd59cdb077e
Fixed threshold: conf >=0.25, NMS IoU=0.70, lesion-matching IoU >=0.50.
All TP/FP/FN are RELATIVE TO ORIGINAL SOURCE ANNOTATIONS, not corrected expert labels.
Expert review indicated possible missing target lesions in some source-negative radiographs.
Ground-truth and prediction boxes are overlaid without modifying source labels.
Panel images derive from publicly released pediatric dental datasets. Confirm respective
licenses and attribution requirements before publication/distribution.
Use only panels for which matching audit passes, and do not claim clinical adjudication.
"""
(OUT/'README_FIGURE_LIMITATIONS.txt').write_text(readme)
zip_path=OUT/'Step_07A_Clinical_Figures_Results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as zz:
 for p in OUT.rglob('*'):
  if p.is_file() and p!=zip_path and 'validation_runs' not in p.parts:
   zz.write(p,p.relative_to(OUT))
print('SUCCESS:',zip_path,'size:',zip_path.stat().st_size)
print('Upload this ZIP to ChatGPT. Do not add any images to a publication before reviewing the panel and licenses.')